In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3 e Qiskit SDK

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK offre alcuni strumenti per convertire tra le rappresentazioni OpenQASM dei programmi quantistici e la classe QuantumCircuit. Tieni presente che questi strumenti sono ancora in una fase esplorativa di sviluppo e continueranno a evolversi man mano che il supporto di Qiskit per le capacità dei circuiti dinamici espresse da OpenQASM 3 aumenta.

> **Note:** Questa funzione è ancora nella fase esplorativa. Pertanto, è probabile che la sintassi e le capacità si evolvano.

## Importare un programma OpenQASM 3 in Qiskit
Devi installare il pacchetto `qiskit_qasm3_import ` per usare questa funzione. Installa con il seguente comando.

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

Attualmente sono disponibili due funzioni di alto livello per importare da OpenQASM 3 in Qiskit. Queste funzioni sono `load()`, che accetta un nome di file, e `loads()`, che accetta il programma stesso come stringa:

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

In questo esempio, definiamo un programma quantistico usando OpenQASM 3 e usiamo `loads()` per convertirlo direttamente in un QuantumCircuit:

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![Output della cella di codice precedente](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## Esportare in OpenQASM 3
Puoi esportare il codice Qiskit in OpenQASM 3 con `dumps()`, che esporta in una stringa, oppure con `dump()`, che esporta in un file.

### Esempio con `dumps()`